# 2. Analysis

**Research question:** is greater internet exposure (weekly `time_spent_hours`) associated with worse ability to identify AI-generated images (`score_pct`)?

Main model: `score_pct ~ time_spent_hours + C(gender) + age`, OLS with HC3 robust standard errors (`src/analysis.py`).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs'
df = pd.read_csv(ROOT / 'data' / 'cleaned_dataset.csv')
A = OUT / 'analysis'
s = json.loads((A / 'summary.json').read_text())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=df, x='time_spent_hours', y='score_pct', hue='gender', ax=axes[0])
sns.regplot(data=df, x='time_spent_hours', y='score_pct', scatter=False, color='k', ax=axes[0])
axes[0].set_title('Time online vs score')
sns.boxplot(data=df, x='gender', y='score_pct', ax=axes[1])
sns.stripplot(data=df, x='gender', y='score_pct', color='k', alpha=0.4, ax=axes[1])
axes[1].set_title('Score by gender')
plt.tight_layout()

## Overall

In [ ]:
t = s['ols']['time_spent_hours']
display(Markdown(f"""- Pearson r = {s['pearson']['r']:.3f} (p = {s['pearson']['pval']:.2f}); Spearman ρ = {s['spearman']['r']:.3f} (p = {s['spearman']['pval']:.2f})
- OLS R² = {s['ols']['r2']:.3f}; 5-fold CV RMSE = {s['ols']['cv_rmse']:.3f}
- Time coefficient: {t['coef']*1000:+.2f} percentage points per 10 h/week (95% CI {t['ci_low']*1000:+.2f} to {t['ci_high']*1000:+.2f}), p = {t['pval']:.2f}"""))
display(pd.read_csv(A / 'pooled_ols_coefficients.csv', index_col=0).round(4))

## By gender

The same question within each gender (`score_pct ~ time_spent_hours + age`), plus the gap between men and women.

In [ ]:
g = s['gender_gap']
display(Markdown(f"""**Gender gap:** men scored {g['male_minus_female_pts']:.1f} percentage points higher than women (Welch t-test p = {g['welch_p']:.3f}; Mann–Whitney p = {g['mannwhitney_p']:.3f}; Cohen's d = {g['cohens_d']:.2f}, a small-to-medium effect)."""))
bg = pd.read_csv(A / 'by_gender.csv').set_index('gender').T
display(bg.round(3))

How to read the table:
- **Time online** has no relationship with score for either gender: the correlations are about zero and the confidence intervals are narrow around zero.
- **Age among men:** the regression shows a negative age effect, about −0.4 points per year. It comes from the few men over 40. Refit on ages ≤ 40, it disappears (`age_pval_under40`), and the rank correlation (`age_spearman_p`) is not significant. Treat it as a hint, not a finding.
- **Age among women:** no effect.

## Robustness

How the time-online effect changes under different modeling choices. Each effect is the change in score (percentage points) for +10 hours a week.

In [ ]:
display(pd.read_csv(A / 'robustness.csv').round(3))

Every specification gives an effect within a few points of zero, and none is significant. Dropping the 130-hour answer moves the estimate slightly positive, which is still the opposite of the hypothesized harm.

## Conclusion

- **No evidence that time online affects the ability to spot AI images**, overall or within either gender. The result holds under every robustness check.
- **Men outscored women by about 5 percentage points.** This is the only robust group difference.
- Average accuracy is only modestly above chance (50%).

**Caveats:** an observational, self-reported convenience sample (mostly ages 16–25), and one imputed hours value. The comparisons are exploratory, with no correction for multiple tests.

## Reproduce
All files read here are generated by the pipeline. From the project root:

```bash
python src/run_all.py
```